# Sentiment Analysis of IMDb Movie Reviews using RNN, LSTM and Word2Vec Embeddings

**Module:** 6CS012 - Artificial Intelligence and Machine Learning  
**Task:** Part III - Natural Language Processing  
**Dataset:** IMDb Movie Reviews (Binary Sentiment Classification)  




## Section 0

In [ ]:
# install everything we need -- only run this once per session
!pip install gensim --quiet
!pip install contractions --quiet
!pip install wordcloud --quiet
!python -c "import nltk; nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)"
print("setup done")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, time, warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
from wordcloud import WordCloud
from collections import Counter

from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")


## Section 1 - Load Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted")


In [ ]:
import zipfile, os

zip_file_path  = '/content/drive/MyDrive/5. Movie Review Dataset-20260503T024216Z-3-001.zip'
extraction_path = '/content/movie_review_dataset'
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as z:
    z.extractall(extraction_path)

print(f'extracted to: {extraction_path}')
print(os.listdir(extraction_path))


In [ ]:
train_df = pd.read_csv('/content/movie_review_dataset/5. Movie Review Dataset/train_movie_review.csv')
val_df   = pd.read_csv('/content/movie_review_dataset/5. Movie Review Dataset/val_movie_review.csv')
test_df  = pd.read_csv('/content/movie_review_dataset/5. Movie Review Dataset/test_movie_review.csv')

# tidy column names
for df in [train_df, val_df, test_df]:
    df.columns = [c.strip().lower() for c in df.columns]

# drop rows with no label
train_df.dropna(subset=['sentiment'], inplace=True)
val_df.dropna(subset=['sentiment'],   inplace=True)
test_df.dropna(subset=['sentiment'],  inplace=True)

# cast to int so keras doesn't complain
for df in [train_df, val_df, test_df]:
    df['sentiment'] = df['sentiment'].astype(int)
    df.reset_index(drop=True, inplace=True)

print(f"Train : {train_df.shape}")
print(f"Val   : {val_df.shape}")
print(f"Test  : {test_df.shape}")
train_df.head(3)


## Section 2 - Data Understanding, Analysis and Visualization

### 2.1 Dataset Description


In [ ]:
total = len(train_df) + len(val_df) + len(test_df)
print(f"Total reviews  : {total}")
print("Task           : Binary sentiment classification (positive / negative)")
print()

for df, name in [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]:
    vc = df['sentiment'].value_counts()
    print(f"{name:6s}  Negative={vc.get(0, 0)}  Positive={vc.get(1, 0)}  Total={len(df)}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#e74c3c', '#2ecc71']

for ax, (df, title) in zip(axes, [(train_df, 'Train Split'),
                                   (val_df,   'Validation Split'),
                                   (test_df,  'Test Split')]):
    counts = df['sentiment'].value_counts().sort_index()
    bars = ax.bar(['Negative', 'Positive'], counts.values, color=colors, width=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 30, str(v),
                ha='center', fontweight='bold')

plt.suptitle('Class Distribution Across Dataset Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
train_df['word_count'] = train_df['review'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_df['word_count'], bins=60, color='steelblue', edgecolor='white')
p95 = int(train_df['word_count'].quantile(0.95))
axes[0].axvline(p95, color='red', linestyle='--', label=f'95th pct = {p95} words')
axes[0].set_title('Review Length Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

for sentiment, label, color in [(0, 'Negative', '#e74c3c'), (1, 'Positive', '#2ecc71')]:
    axes[1].hist(train_df[train_df['sentiment'] == sentiment]['word_count'],
                 bins=40, alpha=0.6, label=label, color=color)
axes[1].set_title('Review Length by Sentiment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('review_length.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean   : {train_df['word_count'].mean():.1f} words")
print(f"Median : {train_df['word_count'].median():.1f} words")
print(f"95th percentile: {p95} words")


## Section 3 - Text Preprocessing, Tokenization and Sequence Padding

### 3.1 Text Cleaning Pipeline


In [ ]:
STOP_WORDS = set(stopwords.words('english'))

NEGATIONS = {'not', 'no', 'nor', 'neither', 'never', 'none', "n't"}
STOP_WORDS = STOP_WORDS - NEGATIONS

lemmatizer = WordNetLemmatizer()

CONTRACTION_MAP = {
    "n't": " not", "'re": " are", "'s": " is", "'d": " would",
    "'ll": " will", "'ve": " have", "'m": " am"
}

def expand_contractions(text):
    for k, v in CONTRACTION_MAP.items():
        text = text.replace(k, v)
    return text

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # strip HTML tags (common in IMDB)
    text = expand_contractions(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

print("Cleaning text...")
t0 = time.time()
train_df['clean'] = train_df['review'].astype(str).apply(clean_text)
val_df['clean']   = val_df['review'].astype(str).apply(clean_text)
test_df['clean']  = test_df['review'].astype(str).apply(clean_text)
print(f"Done in {time.time() - t0:.1f}s")

print("\nBefore:")
print(train_df['review'].iloc[0][:300])
print("\nAfter:")
print(train_df['clean'].iloc[0][:300])


### 3.2 Word Clouds and Top Words per Class

In [ ]:
pos_text = ' '.join(train_df[train_df['sentiment'] == 1]['clean'])
neg_text = ' '.join(train_df[train_df['sentiment'] == 0]['clean'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

wc_pos = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=150).generate(pos_text)
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Positive Reviews - Most Frequent Words', fontsize=13, fontweight='bold')
axes[0].axis('off')

wc_neg = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds', max_words=150).generate(neg_text)
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Negative Reviews - Most Frequent Words', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def top_words(text, n=20):
    words, counts = zip(*Counter(text.split()).most_common(n))
    return list(words), list(counts)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, text, title, color in [
        (axes[0], pos_text, 'Top 20 Words - Positive', '#2ecc71'),
        (axes[1], neg_text, 'Top 20 Words - Negative', '#e74c3c')]:
    words, counts = top_words(text)
    ax.barh(words[::-1], counts[::-1], color=color)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Tokenization and Sequence Padding

In [ ]:
VOCAB_SIZE    = 10000  
OOV_TOKEN     = '<OOV>'
MAX_LEN_RNN   = 100    
MAX_LEN_LSTM  = 150
MAX_LEN_GLOVE = 150

# fit tokenizer on training data only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(train_df['clean'])
word_index = tokenizer.word_index
print(f"Full vocab size : {len(word_index)}")
print(f"Keeping top     : {VOCAB_SIZE} words")

def encode(texts, maxlen):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

X_train_r = encode(train_df['clean'], MAX_LEN_RNN)
X_val_r   = encode(val_df['clean'],   MAX_LEN_RNN)
X_test_r  = encode(test_df['clean'],  MAX_LEN_RNN)

X_train_s = encode(train_df['clean'], MAX_LEN_LSTM)
X_val_s   = encode(val_df['clean'],   MAX_LEN_LSTM)
X_test_s  = encode(test_df['clean'],  MAX_LEN_LSTM)

X_train_g = encode(train_df['clean'], MAX_LEN_GLOVE)
X_val_g   = encode(val_df['clean'],   MAX_LEN_GLOVE)
X_test_g  = encode(test_df['clean'],  MAX_LEN_GLOVE)

y_train = train_df['sentiment'].values
y_val   = val_df['sentiment'].values
y_test  = test_df['sentiment'].values

print(f"\nModel 1 (RNN)   X_train : {X_train_r.shape}")
print(f"Model 2 (LSTM)  X_train : {X_train_s.shape}")
print(f"Model 3 (GloVe) X_train : {X_train_g.shape}")


## Section 4 - Model Building and Training

### 4.1 Shared Settings


In [ ]:
from tensorflow.keras.optimizers import Adam

EMBED_DIM  = 64
UNITS      = 64
BATCH_SIZE = 256
EPOCHS     = 15

def make_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=3,
                      restore_best_weights=True, verbose=1),
        # halve lr if stuck for 2 epochs
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=2, min_lr=1e-5, verbose=1)
    ]

histories   = {}
models      = {}
train_times = {}


### 4.2 Model 1 - Simple RNN with Trainable Embedding

In [ ]:
rnn_model = Sequential(name='Simple_RNN')
rnn_model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                         input_length=MAX_LEN_RNN))
rnn_model.add(SimpleRNN(UNITS, activation='tanh', recurrent_dropout=0.2))
rnn_model.add(Dropout(0.4))  
rnn_model.add(Dense(32, activation='relu'))
rnn_model.add(Dense(1, activation='sigmoid'))  

rnn_model.compile(loss='binary_crossentropy',
                  optimizer=Adam(learning_rate=1e-3),
                  metrics=['accuracy'])
rnn_model.build(input_shape=(None, MAX_LEN_RNN))
rnn_model.summary()


In [ ]:
print("Training Model 1 - Simple RNN")
t0 = time.time()
hist_rnn = rnn_model.fit(
    X_train_r, y_train,
    validation_data=(X_val_r, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1
)
train_times['RNN'] = time.time() - t0
histories['RNN']   = hist_rnn
models['RNN']      = rnn_model
print(f"\nTraining time: {train_times['RNN']:.1f}s")


### 4.3 Model 2 - LSTM with Trainable Embedding

In [ ]:
lstm_model = Sequential(name='LSTM_Trainable_Embed')
lstm_model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                          input_length=MAX_LEN_LSTM))
lstm_model.add(LSTM(UNITS, recurrent_dropout=0.2))
lstm_model.add(Dropout(0.3))
lstm_model.add(Dense(32, activation='relu'))
lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(loss='binary_crossentropy',
                   optimizer=Adam(learning_rate=3e-4), 
                   metrics=['accuracy'])
lstm_model.build(input_shape=(None, MAX_LEN_LSTM))
lstm_model.summary()


In [ ]:
print("Training Model 2 - LSTM (trainable embedding)")
t0 = time.time()
hist_lstm = lstm_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1
)
train_times['LSTM'] = time.time() - t0
histories['LSTM']   = hist_lstm
models['LSTM']      = lstm_model
print(f"\nTraining time: {train_times['LSTM']:.1f}s")


### 4.4 Model 3 - LSTM with Pretrained GloVe Embeddings (Fine-Tuned)

In [ ]:
import gensim.downloader as api

print("Downloading GloVe embeddings (glove-wiki-gigaword-100)...")
embedding_model = api.load('glove-wiki-gigaword-100')
GLOVE_DIM = 100
print("GloVe loaded.")


In [ ]:
embedding_dim    = 100
vocab_size       = VOCAB_SIZE
embedding_matrix = np.zeros((vocab_size, embedding_dim))
hits, misses = 0, 0

for word, i in word_index.items():
    if i >= vocab_size:
        continue
    if word in embedding_model:
        embedding_matrix[i] = embedding_model[word]
        hits += 1
    else:
        misses += 1

print(f"Words found in GloVe : {hits}")
print(f"Words not found      : {misses}")
print(f"Coverage             : {hits / (hits + misses) * 100:.1f}%")


In [ ]:
lstm_w2v = Sequential(name='LSTM_GloVe100_Finetuned')
lstm_w2v.add(Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=MAX_LEN_GLOVE,
    trainable=True
))
lstm_w2v.add(LSTM(UNITS, recurrent_dropout=0.2))
lstm_w2v.add(Dropout(0.3))
lstm_w2v.add(Dense(32, activation='relu'))
lstm_w2v.add(Dense(1, activation='sigmoid'))

lstm_w2v.compile(loss='binary_crossentropy',
                 optimizer=Adam(learning_rate=1e-4),
                 metrics=['accuracy'])
lstm_w2v.build(input_shape=(None, MAX_LEN_GLOVE))
lstm_w2v.summary()


In [ ]:
print("Training Model 3 - LSTM + GloVe-100")
t0 = time.time()
hist_w2v = lstm_w2v.fit(
    X_train_g, y_train,
    validation_data=(X_val_g, y_val),
    epochs=25,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1
)
train_times['LSTM_GloVe'] = time.time() - t0
histories['LSTM_GloVe']   = hist_w2v
models['LSTM_GloVe']      = lstm_w2v
print(f"\nTraining time: {train_times['LSTM_GloVe']:.1f}s")




## Section 5 - Training vs Validation Loss and Accuracy


In [ ]:
def plot_history(hist, title, ax_loss, ax_acc):
    ep = range(1, len(hist.history['loss']) + 1)
    ax_loss.plot(ep, hist.history['loss'],     label='Train Loss', color='steelblue')
    ax_loss.plot(ep, hist.history['val_loss'], label='Val Loss',   color='orange', linestyle='--')
    ax_loss.set_title(title, fontsize=11, fontweight='bold')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend()
    ax_loss.grid(alpha=0.3)

    ax_acc.plot(ep, hist.history['accuracy'],     label='Train Acc', color='steelblue')
    ax_acc.plot(ep, hist.history['val_accuracy'], label='Val Acc',   color='orange', linestyle='--')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_xlabel('Epoch')
    ax_acc.legend()
    ax_acc.grid(alpha=0.3)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
plot_history(hist_rnn,  'Model 1 - Simple RNN',              axes[0][0], axes[1][0])
plot_history(hist_lstm, 'Model 2 - LSTM (Trainable)',        axes[0][1], axes[1][1])
plot_history(hist_w2v,  'Model 3 - LSTM + GloVe (Finetuned)', axes[0][2], axes[1][2])

plt.suptitle('Training vs Validation - Loss and Accuracy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 6 - Model Evaluation on Test Set

### 6.1 Accuracy, Precision, Recall, F1


In [ ]:
def evaluate_model(model, X, y_true, name):
    y_prob = model.predict(X, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)
    print()
    print(f"  {name}")
    print(f"  Test Accuracy : {acc:.4f}")
    print()
    print(classification_report(y_true, y_pred,
                                target_names=['Negative', 'Positive']))
    return y_pred, acc

preds = {}
accs  = {}

preds['RNN'],        accs['RNN']        = evaluate_model(rnn_model,  X_test_r, y_test, 'Model 1 - Simple RNN')
preds['LSTM'],       accs['LSTM']       = evaluate_model(lstm_model, X_test_s, y_test, 'Model 2 - LSTM (Trainable)')
preds['LSTM_GloVe'], accs['LSTM_GloVe'] = evaluate_model(lstm_w2v,  X_test_g, y_test, 'Model 3 - LSTM + GloVe (Finetuned)')


### 6.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_labels = ['Model 1 - Simple RNN',
                'Model 2 - LSTM (Trainable)',
                'Model 3 - LSTM + GloVe (Finetuned)']

for ax, key, title in zip(axes, ['RNN', 'LSTM', 'LSTM_GloVe'], model_labels):
    cm = confusion_matrix(y_test, preds[key])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


### 6.3 Summary Table and Comparison

In [ ]:
rows = []
for key, label in [('RNN', 'Simple RNN'),
                   ('LSTM', 'LSTM (Trainable)'),
                   ('LSTM_GloVe', 'LSTM + GloVe (Finetuned)')]:
    yp = preds[key]
    rows.append({
        'Model'     : label,
        'Accuracy'  : f"{accs[key]:.4f}",
        'Precision' : f"{precision_score(y_test, yp):.4f}",
        'Recall'    : f"{recall_score(y_test, yp):.4f}",
        'F1-Score'  : f"{f1_score(y_test, yp):.4f}",
        'Train Time': f"{train_times[key]:.1f}s"
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

labels    = ['Simple RNN', 'LSTM\n(Trainable)', 'LSTM+GloVe\n(Finetuned)']
acc_vals  = [accs['RNN'], accs['LSTM'], accs['LSTM_GloVe']]
time_vals = [train_times['RNN'], train_times['LSTM'], train_times['LSTM_GloVe']]
colors    = ['#3498db', '#e67e22', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(labels, acc_vals, color=colors, width=0.5, edgecolor='white')
axes[0].set_title('Test Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Accuracy')
for b, v in zip(bars, acc_vals):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.4f}',
                 ha='center', fontweight='bold')

bars2 = axes[1].bar(labels, time_vals, color=colors, width=0.5, edgecolor='white')
axes[1].set_title('Training Time (seconds)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Seconds')
for b, v in zip(bars2, time_vals):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.1f}s',
                 ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 6.4 Results Analysis

**Model 1 - Simple RNN**

50% accuracy, random chance. Recall on positive hit 0.90 while precision sat at 0.50, meaning it predicted positive for nearly everything. Negative class recall was 0.10 -- it caught 1 in 10 actual negatives. F1 macro of 0.41. The model found a shortcut: always guess positive to avoid the worst loss, without learning any real patterns.

**Model 2 - LSTM (Trainable Embedding)**

Accuracy 69.8%. Training was unstable this run -- val_loss bounced around between epochs 3 and 10, the LR was halved twice, and early stopping restored epoch 7 weights. Precision on negative was 0.78 but recall only 0.55, so it missed 45% of actual negatives. Macro F1 of 0.69.

**Model 3 - LSTM + GloVe (Fine-Tuned, 25 epochs)**

Accuracy 77.5%, best of the three. Precision 0.79, recall 0.75 on positive. Negative class: 0.76 precision, 0.80 recall. The gap between precision and recall is narrow on both classes -- the model is not biased toward either label. Macro F1 of 0.77. Training ran 11 epochs before early stopping, restoring weights from epoch 8 (val_loss 0.51).

**Best Model: LSTM + GloVe Fine-Tuned (Model 3)**

Wins on accuracy, F1, and balance. Model 2 underperformed compared to previous runs due to training instability. Model 3's pretrained GloVe starting point gave it more stable gradients from epoch 1.


## Section 7 - Error Analysis


In [ ]:
best_key   = max(accs, key=accs.get)
best_preds = preds[best_key]
print(f"Best model: {best_key}  |  Accuracy: {accs[best_key]:.4f}")
print()

mask   = best_preds != y_test
errors = test_df[mask].copy().reset_index(drop=True)
errors['predicted']  = best_preds[mask]
errors['true_label'] = y_test[mask]

print(f"Misclassified: {len(errors)} / {len(y_test)}  ({len(errors)/len(y_test)*100:.1f}%)")
print()

label_map = {0: 'Negative', 1: 'Positive'}
for i, row in errors.head(3).iterrows():
    print(f"--- Example {i+1} ---")
    print(f"  True label : {label_map[row['true_label']]}")
    print(f"  Predicted  : {label_map[row['predicted']]}")
    print(f"  Review     : {str(row['review'])[:300]}...")
    print()


### 7.1 Misclassified Examples

Best model is LSTM + GloVe with 22.6% error rate (2255 / 10000). All three examples are False Negatives -- positive reviews called negative.

**Example 1 - Purgatory episode (Positive predicted Negative)**

Opens with "I didn't get the Purgatory thing the first time." The model reads the opening as confused or negative and doesn't recover. With max_len=150 and reviews averaging 230 words, the positive conclusion often gets cut off anyway.

**Example 2 - Sharky's Machine (Positive predicted Negative)**

Starts with "Great just great!" which sounds sarcastic in isolation. The reviewer is genuinely positive but the opener followed by plot description gives mixed signals. After cleaning, strong positive words thin out and plot summary words dominate.

**Example 3 - Tragedy film (Positive predicted Negative)**

The reviewer writes "if you are not into tragedies, this is not your movie" and "starts off somewhat slowly." The model sees "not", "slowly", "tragedies" and leans negative. The reviewer is recommending the film but framing it cautiously.

### 7.2 Why Errors Happen

Same pattern across all runs: positive sentiment framed through negative or neutral language. Hedged recommendations, sarcastic openers, and descriptive writing all fool the model the same way.

### 7.3 Suggested Improvements

- **Longer max_len** -- reviews average 230 words but the model only sees 150. Conclusions get cut off regularly.
- **Bidirectional LSTM** -- reading both directions would help weight the end of a review as much as the beginning.
- **Threshold tuning** -- lowering the decision threshold from 0.5 to around 0.43 would recover some missed positives at a small precision cost.
- **More training data** -- 35,000 samples is reasonable but more varied examples of hedged positive reviews would help.


## Section 8 - Real-Time Prediction


In [ ]:
import ipywidgets as widgets
from IPython.display import display

title = widgets.HTML("<h3>IMDB Sentiment Analyser</h3><p>Best model: <b>" + best_key + "</b></p>")
text_input = widgets.Textarea(
    placeholder="Type a movie review here...",
    layout=widgets.Layout(width='600px', height='120px')
)
button = widgets.Button(description="Predict", button_style='primary')
output = widgets.Output()

def on_click(b):
    output.clear_output()
    with output:
        review = text_input.value
        if not review.strip():
            print("Please enter a review.")
            return

        cleaned = clean_text(review)

        if best_key == 'RNN':
            padded = pad_sequences(tokenizer.texts_to_sequences([cleaned]),
                                   maxlen=MAX_LEN_RNN, padding='post', truncating='post')
            model_to_use = rnn_model
        else:
            padded = pad_sequences(tokenizer.texts_to_sequences([cleaned]),
                                   maxlen=MAX_LEN_GLOVE, padding='post', truncating='post')
            model_to_use = models[best_key]

        prob  = float(model_to_use.predict(padded, verbose=0)[0][0])
        label = "Positive" if prob >= 0.5 else "Negative"
        conf  = prob if prob >= 0.5 else 1 - prob

        print(f"Sentiment   : {label}")
        print(f"Confidence  : {conf:.2%}")
        print(f"Raw score   : {prob:.4f}")

button.on_click(on_click)
display(title, text_input, button, output)
